# PharmaOps-AI


# Data Preprocessing

## Objective

The objective of this notebook is to prepare the extracted datasets for analysis by performing data type conversion, validation, consistency checks, and data quality improvements. The processed datasets generated in this notebook will be used for Exploratory Data Analysis (EDA), Feature Engineering, Business Analytics, and Power BI.

In [1]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 1: Connect to MySQL Database

In [2]:
HOST = "localhost"
PORT = 3306
USER = "root"
PASSWORD = "Venkat#26158"
DATABASE = "pharmaops_ai"

engine = create_engine(
    f"mysql+mysqlconnector://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)

print("Database connection established successfully.")

Database connection established successfully.


## Step 2: Load Datasets

In [3]:
# Medicines Master from processed CSV
medicines_df = pd.read_csv("../../data/processed/Medicines_Master_Final.csv")

# Remaining tables from MySQL
inventory_df = pd.read_sql("SELECT * FROM Medicines_Inventory", engine)
suppliers_df = pd.read_sql("SELECT * FROM Suppliers_Master", engine)
sales_df = pd.read_sql("SELECT * FROM Sales_Transactions", engine)
waste_df = pd.read_sql("SELECT * FROM Waste_Records", engine)
category_df = pd.read_sql("SELECT * FROM Category_Master", engine)

datasets = {
    "Medicines_Master": medicines_df,
    "Medicines_Inventory": inventory_df,
    "Suppliers_Master": suppliers_df,
    "Sales_Transactions": sales_df,
    "Waste_Records": waste_df,
    "Category_Master": category_df
}

print("All datasets loaded successfully.")

All datasets loaded successfully.


## Step 3: Convert Date Columns

Convert all date-related columns to datetime format for accurate analysis and feature engineering.

In [4]:
# Medicines Inventory
inventory_df["Manufacturing_Date"] = pd.to_datetime(inventory_df["Manufacturing_Date"])
inventory_df["Expiry_Date"] = pd.to_datetime(inventory_df["Expiry_Date"])
inventory_df["Last_Restock_Date"] = pd.to_datetime(inventory_df["Last_Restock_Date"])

# Suppliers
suppliers_df["Contract_Start_Date"] = pd.to_datetime(suppliers_df["Contract_Start_Date"])
suppliers_df["Contract_End_Date"] = pd.to_datetime(suppliers_df["Contract_End_Date"])

# Sales
sales_df["Transaction_Date"] = pd.to_datetime(sales_df["Transaction_Date"])

# Waste
waste_df["Waste_Date"] = pd.to_datetime(waste_df["Waste_Date"])
waste_df["Expiry_Date"] = pd.to_datetime(waste_df["Expiry_Date"])

print("Date columns converted successfully.")

Date columns converted successfully.


In [5]:
# Integer columns

inventory_df = inventory_df.astype({
    "Quantity_In_Stock":"int32",
    "Reorder_Level":"int16"
})

sales_df = sales_df.astype({
    "Quantity_Sold":"int16"
})

waste_df = waste_df.astype({
    "Quantity_Wasted":"int16"
})

suppliers_df = suppliers_df.astype({
    "Lead_Time_Days":"int16"
})

# Float columns

inventory_df = inventory_df.astype({
    "Unit_Cost":"float32",
    "Selling_Price":"float32"
})

sales_df = sales_df.astype({
    "Unit_Selling_Price":"float32",
    "Discount_Percentage":"float32",
    "Total_Amount":"float32"
})

waste_df = waste_df.astype({
    "Unit_Cost":"float32",
    "Total_Waste_Value":"float32"
})

suppliers_df = suppliers_df.astype({
    "Supplier_Rating":"float32"
})

print("Numeric data types optimized successfully.")

Numeric data types optimized successfully.


In [6]:
print("Inventory Quantity < 0 :",
      (inventory_df["Quantity_In_Stock"] < 0).sum())

print("Selling Price < Unit Cost :",
      (inventory_df["Selling_Price"] < inventory_df["Unit_Cost"]).sum())

print("Expired Before Manufacturing :",
      (inventory_df["Expiry_Date"] <
       inventory_df["Manufacturing_Date"]).sum())

print("Negative Waste Quantity :",
      (waste_df["Quantity_Wasted"] < 0).sum())

print("Negative Sales Quantity :",
      (sales_df["Quantity_Sold"] < 0).sum())

Inventory Quantity < 0 : 0
Selling Price < Unit Cost : 0
Expired Before Manufacturing : 0
Negative Waste Quantity : 0
Negative Sales Quantity : 0


In [7]:
print("Medicines without Category")

print(
    medicines_df["Category_ID"]
    .isin(category_df["Category_ID"])
    .value_counts()
)

print()

print("Inventory Medicines Validation")

print(
    inventory_df["Medicine_ID"]
    .isin(medicines_df["Medicine_ID"])
    .value_counts()
)

print()

print("Sales Medicines Validation")

print(
    sales_df["Medicine_ID"]
    .isin(medicines_df["Medicine_ID"])
    .value_counts()
)

print()

print("Waste Medicines Validation")

print(
    waste_df["Medicine_ID"]
    .isin(medicines_df["Medicine_ID"])
    .value_counts()
)

Medicines without Category
Category_ID
True    111815
Name: count, dtype: int64

Inventory Medicines Validation
Medicine_ID
True    4954
Name: count, dtype: int64

Sales Medicines Validation
Medicine_ID
True    9855
Name: count, dtype: int64

Waste Medicines Validation
Medicine_ID
True    2481
Name: count, dtype: int64


In [8]:
text_columns = [
    (medicines_df,["Generic_Name","Brand_Name","Manufacturer"]),
    (category_df,["Category_Name"]),
    (suppliers_df,["Supplier_Name","City","State"]),
]

for df,cols in text_columns:

    for col in cols:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
        )

print("Text standardization completed.")

Text standardization completed.


In [9]:
summary = pd.DataFrame({

    "Dataset":[
        "Medicines",
        "Inventory",
        "Suppliers",
        "Sales",
        "Waste",
        "Category"
    ],

    "Rows":[
        len(medicines_df),
        len(inventory_df),
        len(suppliers_df),
        len(sales_df),
        len(waste_df),
        len(category_df)
    ],

    "Columns":[
        medicines_df.shape[1],
        inventory_df.shape[1],
        suppliers_df.shape[1],
        sales_df.shape[1],
        waste_df.shape[1],
        category_df.shape[1]
    ]

})

summary

,Dataset,Rows,Columns
0,Medicines,111815,11
1,Inventory,4954,14
2,Suppliers,75,15
3,Sales,9855,15
4,Waste,2481,15
5,Category,15,3


In [10]:
output_path="../../data/processed/python/"

medicines_df.to_csv(output_path+"Medicines.csv",index=False)

inventory_df.to_csv(output_path+"Inventory.csv",index=False)

suppliers_df.to_csv(output_path+"Suppliers.csv",index=False)

sales_df.to_csv(output_path+"Sales.csv",index=False)

waste_df.to_csv(output_path+"Waste.csv",index=False)

category_df.to_csv(output_path+"Category.csv",index=False)

print("Processed datasets exported successfully.")

Processed datasets exported successfully.
